# Document Processing with Unstructured Transform MCP

In this recipe, we connect to [Unstructured Transform](https://docs.unstructured.io/transform/overview)'s hosted MCP server and use a Haystack `Agent` to parse and chunk a document, entirely through MCP tools.

**Services used:**
- [Unstructured Transform MCP](https://mcp.transform.unstructured.io): document processing (partition, enrich, chunk, embed) exposed as MCP tools
- [Anthropic Claude](https://www.anthropic.com/): LLM for agent reasoning

## Install dependencies

In [ ]:
!pip install -q haystack-ai mcp-haystack anthropic-haystack

## Set up API keys

You'll need two API keys:
- **Unstructured API key**: get one from the [Transform get-started page](https://transform.unstructured.io/get-started) after signing in. The free tier includes 15,000 pages a month.
- **Anthropic API key**: get one at [console.anthropic.com](https://console.anthropic.com/)

In [ ]:
import os
from getpass import getpass

if "UNSTRUCTURED_API_KEY" not in os.environ:
    os.environ["UNSTRUCTURED_API_KEY"] = getpass("Enter your Unstructured API key: ")
if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

## Step 1: Connect to Unstructured Transform MCP

[Unstructured Transform](https://docs.unstructured.io/transform/overview) exposes its document-processing pipeline (partition, enrich, chunk, embed) as a hosted MCP server at `https://mcp.transform.unstructured.io`. We connect to it with [`MCPToolset`](https://docs.haystack.deepset.ai/docs/mcptoolset), using `StreamableHttpServerInfo`'s native `token` parameter to send the Unstructured API key as an `Authorization: Bearer` header.

In [ ]:
from haystack_integrations.tools.mcp import MCPToolset, StreamableHttpServerInfo
from haystack.utils import Secret

server_info = StreamableHttpServerInfo(
    url="https://mcp.transform.unstructured.io",
    token=Secret.from_env_var("UNSTRUCTURED_API_KEY"),
)
toolset = MCPToolset(server_info=server_info, eager_connect=True)

for tool in toolset.tools:
    print(f"{tool.name}: {tool.description}")

This exposes four tools:
- `start_transform_job`: submits one or more files for processing and returns a `job_id` right away; the job itself runs asynchronously
- `check_job_status`: polls a job's status until it reaches `COMPLETED`
- `get_job_results`: fetches the rendered output of a completed job, as markdown, JSON, HTML, or plain text
- `request_file_upload_url`: returns a presigned upload URL for a local file, for inputs that aren't already reachable over HTTPS

## Step 2: Build a Haystack Agent with the Transform MCP toolset

We give the agent the toolset directly, along with a system prompt describing the asynchronous `start_transform_job` -> `check_job_status` -> `get_job_results` flow, since the agent needs to poll for a result rather than get one back immediately.

In [ ]:
from haystack.components.agents import Agent
from haystack_integrations.components.generators.anthropic import AnthropicChatGenerator

agent = Agent(
    chat_generator=AnthropicChatGenerator(
        model="claude-opus-4-6",
        generation_kwargs={"max_tokens": 4096},
    ),
    tools=toolset,
    system_prompt="""You are a document-processing assistant with access to Unstructured Transform MCP tools.

Transform jobs are asynchronous. When asked to process a document:
1. Call `start_transform_job` with the file reference(s) and the requested processing stages. This returns a `job_id` immediately; the job itself runs in the background.
2. Call `check_job_status` with that `job_id`, repeating until the status is COMPLETED.
3. Call `get_job_results` with the `job_id` to fetch the rendered output, and summarize it for the user.
""",
)

## Step 3: Process a document end-to-end

We hand the agent a publicly reachable PDF and ask it to parse and chunk it. No local file or upload step is needed here, since `start_transform_job` accepts `https://` URLs directly.

In [ ]:
from haystack.dataclasses import ChatMessage

pdf_url = "https://arxiv.org/pdf/1706.03762"

result = agent.run(
    messages=[
        ChatMessage.from_user(
            f"Parse and chunk the PDF at {pdf_url}. "
            "Use the 'hi_res' partition strategy, and chunk with chunk_by_title, "
            "max_characters=1000. Once the job is complete, fetch the results as "
            "markdown and show me the first two chunks."
        )
    ]
)

In [ ]:
print(result["last_message"].text)

### A note on the OAuth fallback

Above, we authenticated with a static Unstructured API key, the simplest path for headless agent frameworks like this one. Transform MCP also supports interactive OAuth/OIDC login for clients that speak remote MCP natively.

If your MCP client only supports local (stdio) servers, or doesn't support browser-based OAuth for remote servers, you can bridge to the hosted server with [`mcp-remote`](https://www.npmjs.com/package/mcp-remote):

```bash
npm install -g mcp-remote
npx -y mcp-remote https://mcp.transform.unstructured.io
```

This runs a local stdio proxy that handles the browser OAuth flow and forwards requests to `https://mcp.transform.unstructured.io`.

## Conclusion

Unstructured Transform's MCP server brings partitioning, enrichment, chunking, and embedding into a single set of tools an agent can call directly, without wiring up a separate ETL pipeline. Because `start_transform_job` runs asynchronously and returns a `job_id` right away, a Haystack `Agent` can poll for completion and fetch results the same way it would call any other tool, making it straightforward to drop document processing into a larger agentic workflow.